# Oscillatory 3D producing-well flow

This notebook displays native hexahedral RT1/Q1 MHM results for the tensor field of L05 Problem 5 on an explicitly declared polygonal reservoir. The reference is the classical conforming fine RT1 method with its own refinement study, not an exact field. Historical graded radii and the printed outer permeability scale are ambiguous.


## Physical data

The tensor is $K=10^{-11}\operatorname{diag}(\kappa_x,\kappa_y,\kappa_z)$ m², with the trigonometric ratios stated in the case documentation. Every factor $2+1.8\sin(\cdot)$ lies in $[0.2,3.8]$, so no permeability floor is used. The scale matches Figure 16 ranges; the printed factor is $10^{-3}$. Positive $Q$ denotes production. Pressure is imposed at both lateral walls and top/bottom fluxes are zero. The actual heterogeneous total rate is measured.

The horizontal tensor and boundary data are independent of height, and the impermeable caps admit qz=0. The classical reference therefore refines (F,F,1) with the full three-dimensional RT1/Q1 operator; a separate vertically refined comparison checks this invariance. The MHM cases retain the isotropic 3D fine grid.


The largest classical systems use the exact vertically invariant subspace: 12 horizontal RT1 flux modes and four height-independent Q1 pressure modes per cell, with the full three-dimensional Piola map and volume measure. The reduction is verified against complete 3D systems. All distributed coefficient archives are float64, and their physical equations are checked after conversion. This is a native PyMHM classical reference, not an independent external solver.


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

root = Path.cwd()
if not (root / "examples").is_dir():
    root = root.parent
records = root / "examples/results/mapped-well-oscillatory"
report = json.loads((records / "comparisons.json").read_text())
for row in report["rows"]:
    order = str(max(map(int, row["norms"])))
    norm = row["norms"][order]
    print(row["kind"], row["candidate"], "versus", row["reference"],
          "pressure increment difference [%]", 100 * norm["pressure_increment_relative"],
          "total pressure difference [%]", 100 * norm["pressure_relative"],
          "flux difference [%]", 100 * norm["flux_relative"])


## Fixed macro traces and local refinement

The F16 control keeps the 2,048 macro hexahedra and their RT1 normal-trace coefficients fixed, while increasing fine hexahedra from 16,384 to 131,072. Its comparison with classical F16 isolates the trace restriction at the richer local resolution. Its comparison with the finest classical reference still contains fine approximation and trace effects. Each normalized difference uses the explicitly named reference field; percentages with different denominators are not added. Material assembly orders 28 and 40 are checked independently. The table reads the highest recorded physical-norm quadrature for every pair.


## Physical fields and interfaces

Pressure and flux are sampled independently on each fine element. The same MHM macro partition is drawn on reference, MHM and error panels. Common scales compare the two physical fields; differences have separate scales. Display samples do not replace integrated physical norms.

Signed horizontal flux components use an odd asinh color normalization, with its linear width set to one ten-thousandth of the full absolute range. Each component shares a scale between reference and MHM; its signed difference uses a separate scale. Colorbar ticks remain in physical units. No values are clipped. The invariant reference has zero vertical flux; the component metadata also records the sampled vertical MHM flux.


In [ ]:
figures = root / "docs/figures/mapped-well-oscillatory"
for name in ["material", "macro1", "macro2", "macro4", "macro4-fine16", "local-resolution", "flux-logarithmic", "flux-logarithmic-fine16", "flux-components-fine8", "flux-components-fine16", "profiles", "convergence"]:
    display(Image(filename=str(figures / f"{name}.png")))


## Interpretation

Material quadrature, coarse normal-trace restriction and classical spatial refinement are distinct error sources. The norm tables preserve those distinctions and report a quadrature check for volume integration. Residuals and pressure-tested conservation moments verify the discrete equations; they do not by themselves establish accuracy, stability for arbitrary maps, or historical reproduction.

At fixed fine resolution F8, releasing more macro trace moments reduces the MHM-to-classical-F8 flux difference from 37.2872% to 20.8571% and 11.6196%. The corresponding pressure-increment differences are 41.3800%, 9.19376% and 2.84815%. These ratios use F8 norms. Integration retains every vertical MHM cell and its computed vertical flux.

The following summary uses the finest classical reference named in the comparison record. The direct classical F8-to-reference difference measures fine-space sensitivity; the MHM-to-reference differences also contain coarse-trace restriction. Ratios with different reference denominators are not additive, and no orthogonal error decomposition is assumed.

The pressure-increment denominator uses $p-25\,\mathrm{MPa}$ so that the large ambient pressure does not hide changes in the drawdown. The final classical increment measures reference sensitivity; it is not a guaranteed bound on exact-solution error.


In [ ]:
reference = report["reference"]
reference_record = json.loads((records / f"{reference}.json").read_text())
print("Classical reference:", reference)
print("Fine hexahedra:", reference_record["fine_cells"],
      "mixed unknowns:", reference_record["unknowns"])
for row in report["rows"]:
    if row["reference"] != reference or row["kind"] not in {
        "classical spatial refinement", "classical fine-resolution gap",
        "MHM versus refined reference", "local-control refined reference",
    }:
        continue
    order = str(max(map(int, row["norms"])))
    norm = row["norms"][order]
    print(row["kind"], row["candidate"], "quadrature", norm["quadrature_order"],
          "flux [%]", 100 * norm["flux_relative"],
          "pressure increment [%]", 100 * norm["pressure_increment_relative"])


## Independently assembled matching systems

DOLFINx 0.9.0/UFL and Basix assemble the same annular geometry, anisotropic material and boundary-value problem. Classical RT1/Q1 fields at F2, F8 and F16 release all fine normal moments; an additional full three-dimensional F2 solve verifies height invariance. For the three MHM macro partitions at F8, the independent global mixed operator retains the same macroface P1 normal subspace through native Basix interpolation. This is a restricted conforming mixed solve, not a native MHM controller.

The differences below integrate the complete three-dimensional physical volume. Every vertical PyMHM cell and its computed vertical flux are retained. The pressure-increment denominator is the independent norm of $p-25\,\mathrm{MPa}$. The figure samples the first vertical layer's middle Gauss plane, $z=-4.375$ m, with actual macro intersections and separate finite-precision difference scales. Agreement verifies these discrete systems; it does not imply that their coarse approximation is accurate or identify the ambiguous historical mesh and permeability scale.


In [ ]:
verification = json.loads((records / "native-discrete-verification.json").read_text())
assert verification["accepted"] and len(verification["rows"]) == 7
for row in verification["rows"]:
    comparison = max(row["comparisons"], key=lambda item: item["order"])
    print(row["case"], "relative flux difference", comparison["flux_relative"],
          "relative pressure-increment difference", comparison["pressure_increment_relative"])
display(Image(filename=str(figures / "native-comparison.png")))
